# Neural Network Interpretability

> Based on Stanford CS230 — LN9

Deep models are often treated as black boxes, yet understanding *what* they learn is critical for debugging, safety, and trust. Interpretability methods can be **local** (explain one prediction) or **global** (explain overall model behaviour).

## Learning Objectives

1. Compute **saliency maps** via gradient of output w.r.t. input
2. Implement **Grad-CAM**-style class activation mapping
3. Apply **LIME**-style local linear approximation
4. Visualise **feature attribution** with integrated gradients concept

## Methods at a Glance

| Method | Type | How it works |
|---|---|---|
| Saliency map | Local | $\|\nabla_x \hat{y}_c\|$ — which pixels affect output most |
| Grad-CAM | Local | Weighted sum of feature maps using gradient signal |
| Integrated Gradients | Local | Integral of gradients along path from baseline to input |
| LIME | Local | Fit interpretable model on perturbed neighbourhood |
| Activation Maximisation | Global | $\arg\max_x \hat{y}_c(x)$ — what input maximises a neuron |

## Saliency Maps

For a classifier output $S_c(x)$:

$$M_{ij} = \left|\frac{\partial S_c}{\partial x_{ij}}\right|$$

Large values indicate pixels the network is sensitive to — but they highlight *where* the model looks, not *why*.

## Integrated Gradients

$$\text{IG}_i(x) = (x_i - x'_i) \cdot \int_0^1 \frac{\partial F(x' + \alpha(x-x'))}{\partial x_i}\, d\alpha$$

where $x'$ is the baseline (e.g., black image). This satisfies the **completeness axiom**: attributions sum to $F(x) - F(x')$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

plt.rcParams.update({
    'figure.facecolor': '#f5f5f7', 'axes.facecolor': '#ffffff',
    'axes.edgecolor': '#c8ccd4', 'axes.labelcolor': '#1a1d27',
    'axes.titlecolor': '#1a1d27', 'xtick.color': '#2a2e3a',
    'ytick.color': '#2a2e3a', 'grid.color': '#e0e3ea',
    'grid.linestyle': '--', 'grid.alpha': 0.5,
    'text.color': '#1a1d27', 'font.family': 'DejaVu Sans',
    'axes.titlesize': 12, 'axes.labelsize': 11,
    'legend.facecolor': '#ffffff', 'legend.edgecolor': '#c8ccd4',
    'figure.dpi': 110,
})
P = ['#5b9bd5', '#e05c5c', '#f4b942', '#7ecba1', '#56b6c2', '#c678dd']
saliency_cmap = LinearSegmentedColormap.from_list('sal', ['#ffffff', '#f4b942', '#e05c5c'])

def relu(z): return np.maximum(0, z)
def relu_d(z): return (z > 0).astype(float)
def softmax(z):
    e = np.exp(z - z.max())
    return e / e.sum()

# ---- Toy CNN-like model on a 6×6 "image" ----
np.random.seed(7)
H, W = 6, 6
n_pixels = H * W
n_h, n_y = 24, 3

W1 = np.random.randn(n_h, n_pixels) * 0.15
b1 = np.zeros(n_h)
W2 = np.random.randn(n_y, n_h) * 0.15
b2 = np.zeros(n_y)

def forward_and_cache(x_flat):
    z1 = W1 @ x_flat + b1
    a1 = relu(z1)
    z2 = W2 @ a1 + b2
    probs = softmax(z2)
    return probs, a1, z1

# ---- Saliency map: ∂S_c/∂x ----
def saliency(x_flat, target_class):
    probs, a1, z1 = forward_and_cache(x_flat)
    # Backprop to input
    d_out      = np.zeros(n_y); d_out[target_class] = 1.0
    d_z2       = probs * (d_out - probs.dot(d_out))  # softmax backprop
    d_a1       = W2.T @ d_z2
    d_z1       = d_a1 * relu_d(z1)
    d_x        = W1.T @ d_z1
    return np.abs(d_x).reshape(H, W), probs

# ---- Integrated Gradients ----
def integrated_gradients(x_flat, baseline, target_class, n_steps=50):
    alphas = np.linspace(0, 1, n_steps)
    grads  = []
    for a in alphas:
        xi = baseline + a * (x_flat - baseline)
        _, a1, z1 = forward_and_cache(xi)
        probs_i = softmax(W2 @ relu(z1) + b2)
        d_out = np.zeros(n_y); d_out[target_class] = 1.0
        d_z2  = probs_i * (d_out - probs_i.dot(d_out))
        d_x   = W1.T @ (relu_d(z1) * (W2.T @ d_z2))
        grads.append(d_x)
    avg_grad = np.array(grads).mean(axis=0)
    ig = (x_flat - baseline) * avg_grad
    return ig.reshape(H, W)

# ---- Activation maximisation (gradient ascent on input) ----
def activation_maximise(target_class, n_iter=200, lr=0.05, l2=0.01, seed=0):
    np.random.seed(seed)
    x = np.random.randn(n_pixels) * 0.01
    for _ in range(n_iter):
        probs, a1, z1 = forward_and_cache(x)
        d_out = np.zeros(n_y); d_out[target_class] = 1.0
        d_z2  = probs * (d_out - probs.dot(d_out))
        grad  = W1.T @ (relu_d(z1) * (W2.T @ d_z2))
        x    += lr * (grad - l2 * x)
    return x.reshape(H, W)

# Create a synthetic "image" with a clear pattern for each class
def make_image(class_id, seed=42):
    np.random.seed(seed + class_id)
    img = np.random.randn(n_pixels) * 0.1
    # Plant a class-specific region
    patch = slice(class_id*2, class_id*2+2)
    img.reshape(H, W)[patch, :] += 1.5
    return img

# ---- Plot ----
fig, axes = plt.subplots(3, 4, figsize=(16, 11))
fig.suptitle('Neural Network Interpretability Methods', fontsize=13, fontweight='bold')

for cls in range(3):
    x_img = make_image(cls)
    probs, _, _ = forward_and_cache(x_img)
    pred = probs.argmax()

    sal,    _ = saliency(x_img, cls)
    ig_map    = integrated_gradients(x_img, np.zeros(n_pixels), cls)
    act_max   = activation_maximise(cls)

    # Input image
    axes[cls, 0].imshow(x_img.reshape(H, W), cmap='gray', vmin=-2, vmax=2)
    axes[cls, 0].set_title(f'Input (class {cls})\npred={pred}, conf={probs[cls]:.2f}')
    axes[cls, 0].axis('off')

    # Saliency
    axes[cls, 1].imshow(sal, cmap=saliency_cmap, vmin=0)
    axes[cls, 1].set_title(f'Saliency Map\n|∂S_{cls}/∂x|')
    axes[cls, 1].axis('off')

    # Integrated Gradients
    ig_abs = np.abs(ig_map)
    axes[cls, 2].imshow(ig_abs, cmap=saliency_cmap, vmin=0)
    axes[cls, 2].set_title(f'Integrated Gradients\n(sum={ig_map.sum():.3f})')
    axes[cls, 2].axis('off')

    # Activation Maximisation
    axes[cls, 3].imshow(act_max, cmap='RdBu_r', vmin=-act_max.std()*2, vmax=act_max.std()*2)
    axes[cls, 3].set_title(f'Activation Maximisation\n(class {cls})')
    axes[cls, 3].axis('off')

col_titles = ['Input Image', 'Saliency Map', 'Integrated Gradients', 'Activation Max.']
for ax, t in zip(axes[0], col_titles):
    ax.set_title(t + '\n' + ax.get_title().split('\n')[1], fontsize=10)

plt.tight_layout()
plt.show()

# ---- LIME-style: local linear approximation ----
np.random.seed(10)
n_samples = 200
x0       = make_image(0)
noise    = np.random.randn(n_samples, n_pixels) * 0.5
X_perturbed = x0 + noise
y_perturbed = np.array([forward_and_cache(x)[0][0] for x in X_perturbed])

# Fit a linear model (closed-form ridge regression)
lam = 0.1
A   = X_perturbed.T @ X_perturbed + lam * np.eye(n_pixels)
b_  = X_perturbed.T @ y_perturbed
lime_coeff = np.linalg.solve(A, b_).reshape(H, W)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(lime_coeff, cmap='RdBu_r', vmin=-np.abs(lime_coeff).max(), vmax=np.abs(lime_coeff).max())
ax.set_title('LIME-style Local Attribution\n(linear proxy coefficients for class 0)', fontsize=11)
ax.axis('off')
plt.colorbar(im, ax=ax, label='attribution weight')
plt.tight_layout()
plt.show()
